In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
print('Version of tensorflow :',tf.__version__)

Version of tensorflow : 2.15.0


In [3]:
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from keras.optimizers import Adam
from keras.applications.vgg19 import VGG19,preprocess_input
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import glob
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from datetime import datetime
from keras.callbacks import ModelCheckpoint

In [4]:
IMAGE_SIZE = [ 224 , 224 , 3 ]

# Load the model
vgg = VGG19( include_top = False,
            input_shape = IMAGE_SIZE,
            weights = 'imagenet')

80134624/80134624 [==============================] - 0s 0us/step


In [5]:
for  layer in vgg.layers:
    layer.trainable = False

In [6]:
# Flattened the last layer
x = Flatten()(vgg.output)

# Created a new layer as output
prediction = Dense( 7 , activation = 'softmax' )(x)

# Join it with the model
model = Model( inputs = vgg.input , outputs = prediction )

# Visualize the model again
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [7]:
from keras.layers import Dense
from keras.models import Model

# Assuming 'num_classes' is the number of classes in your classification task
num_classes = 7  # Change this to the correct number of classes

# Remove the existing output layer
model_output = model.layers[-2].output  # Get the output of the second-to-last layer
output_layer = Dense(num_classes, activation='softmax')(model_output)

# Create a new model with the adjusted output layer
new_model = Model(inputs=model.input, outputs=output_layer)

# Verify the updated model architecture
new_model.summary()


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0   

In [8]:
from tensorflow.keras.optimizers import RMSprop

# Define RMSprop optimizer with a smaller learning rate
rmsprop = RMSprop(learning_rate=0.001)  # Adjust the learning rate as needed

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])


In [9]:
# For the train data generator
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode='nearest'
)

# For the test data generator
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode='nearest'
)


In [10]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test'

In [11]:
# train data
train_set = train_datagen.flow_from_directory(train_path,
                                            target_size = ( 224 , 224 ),
                                            batch_size = 32,
                                            class_mode = 'categorical')

# test data
test_set = test_datagen.flow_from_directory(test_path,
                                             target_size = ( 224 , 224 ),
                                            batch_size = 32,
                                            class_mode = 'categorical')

Found 6300 images belonging to 7 classes.
Found 1580 images belonging to 7 classes.


In [12]:
# Inspect a few samples from the training set
for images, labels in train_set:
    print('Images shape:', images.shape)
    print('Labels shape:', labels.shape)
    print('Labels:', labels)
    break  # Break after the first batch to avoid printing too many samples

# Inspect a few samples from the test set
for images, labels in test_set:
    print('Images shape:', images.shape)
    print('Labels shape:', labels.shape)
    print('Labels:', labels)
    break  # Break after the first batch to avoid printing too many samples


Images shape: (32, 224, 224, 3)
Labels shape: (32, 7)
Labels: [[0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]]
Images shape: (32, 224, 224, 3)
Labels shape: (32, 7)
Labels: [[1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 

In [ ]:
from matplotlib import image as mpimg
import os
for link in os.listdir(train_path):
  lk=train_path+'/'+link
  for lk1 in os.listdir(lk+'/'):
    path=lk+'/'+lk1
    img=mpimg.imread(path)
    plt.imshow(img)
    plt.title(link[:-1])
    plt.show()
    break

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
output_layer = model.layers[-1]  # Assuming the output layer is the last layer in the model
num_classes = output_layer.output_shape[-1]  # Number of units in the output layer

print("Number of classes in the output layer:", num_classes)


Number of classes in the output layer: 7


In [ ]:
checkpoint = ModelCheckpoint(filepath = '/content/drive/MyDrive/Models/vgg19.h5' , verbose = 2 , save_best_only = True )
callbacks = [checkpoint]
start = datetime.now()
model_history = model.fit( train_set,
                          validation_data = test_set,
                          epochs = 10,
                          steps_per_epoch = 197,
                          validation_steps = 50,
                          callbacks = callbacks,)

duration = datetime.now() - start

print('Total elapsed time : ',duration)

Epoch 1/10
197/197 [==============================] - ETA: 0s - loss: 4.9282 - accuracy: 0.7490 
Epoch 1: val_loss improved from inf to 2.71627, saving model to /content/drive/MyDrive/Models/vgg19.h5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


197/197 [==============================] - 3280s 17s/step - loss: 4.9282 - accuracy: 0.7490 - val_loss: 2.7163 - val_accuracy: 0.8557
Epoch 2/10
197/197 [==============================] - ETA: 0s - loss: 0.7411 - accuracy: 0.9400
Epoch 2: val_loss improved from 2.71627 to 1.78390, saving model to /content/drive/MyDrive/Models/vgg19.h5
197/197 [==============================] - 806s 4s/step - loss: 0.7411 - accuracy: 0.9400 - val_loss: 1.7839 - val_accuracy: 0.8962
Epoch 3/10
197/197 [==============================] - ETA: 0s - loss: 0.5290 - accuracy: 0.9594
Epoch 3: val_loss did not improve from 1.78390
197/197 [==============================] - 568s 3s/step - loss: 0.5290 - accuracy: 0.9594 - val_loss: 1.9593 - val_accuracy: 0.8918
Epoch 4/10
197/197 [==============================] - ETA: 0s - loss: 0.3466 - accuracy: 0.9713
Epoch 4: val_loss did not improve from 1.78390
197/197 [==============================] - 536s 3s/step - loss: 0.3466 - accuracy: 0.9713 - val_loss: 2.4665 - va

# HyperParameter Tuning


In [ ]:
!ls /content/drive/My\ Drive/


 AnjnayMahajan_Mechanical_Mainstream.pdf  'Colab Notebooks'


In [ ]:
!find /content/drive/My\ Drive/ -name "vgg19.h5"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# List contents of the /content/ directory to find the model file
print("Contents of the /content/ directory:")
!ls /content/

# Check if the model file exists in /content/
model_file_path = '/content/vgg19.h5'
if os.path.exists(model_file_path):
    print(f"Model saved at: {model_file_path}")
else:
    print("Model not found in /content/")

# If the model file exists, move it to Google Drive
destination_path = '/content/drive/My Drive/models/vgg19.h5'
if os.path.exists(model_file_path):
    # Create the destination directory if it doesn't exist
    os.makedirs('/content/drive/My Drive/models', exist_ok=True)
    # Move the file to Google Drive
    !mv /content/vgg19.h5 /content/drive/My\ Drive/models/
    print(f"Model moved to: {destination_path}")
else:
    print("Model file not found, nothing to move.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contents of the /content/ directory:
drive  sample_data
Model not found in /content/
Model file not found, nothing to move.


In [ ]:
from tensorflow.keras.models import load_model

# Load the model
model2 = load_model()

TypeError: load_model() missing 1 required positional argument: 'filepath'